# Evaluating frontier LLMs

In [1]:
import os
import time
from dotenv import load_dotenv
from pathlib import Path
import numpy as np
from datetime import datetime
import json
import pandas as pd
from openai import OpenAI
import anthropic
from pydantic import BaseModel
from typing import Literal
import pickle

data_path = "../data"

with open(f"{data_path}/test_set.json", "r") as f:
    test_set = json.load(f)

In [2]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')

ANTHROPIC_BASE_URL = "https://api.anthropic.com/v1/"

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set (and this is optional)")

openai_client = OpenAI()
anthropic_client = anthropic.Anthropic()

OpenAI API Key exists and begins sk-proj-
Anthropic API Key exists and begins sk-ant-


In [3]:
models = {
    "openai": [
        "gpt-5-nano", 
        "gpt-6-astra"
        ],
    "anthropic": [
        "claude-haiku-4-5-20251001", 
        "claude-fable-5-1"
        ]
}

### Transform data into appropriate format

In [5]:
def prompt_transform(target_set, test):

    modified_set = []

    for item in target_set:
        headlines = "\n\n".join([f"{i + 1}: {headline}" for i, headline in enumerate(item["headlines"])])
        headlines = f"Identify which of the four following headlines is fake:\n\n{headlines}\n\nAnswer: "
        if test:
            entry = {
                "date": item["date"],
                "prompt": headlines,
                "answer": item["answer"],
            }
        else:
            entry = {
                "date": item["date"],
                "prompt": headlines,
                "answer": item["answer"],
                "completion": str(item["answer"] + 1)
            }

        modified_set.append(entry)

    return modified_set

test_set_frontier_LLM_ready = prompt_transform(test_set, True)

### Setup and abstraction function

Create structured output

In [6]:
class FakeHeadlineOutput(BaseModel):
    """Structured numeric answer for predicting which headline is fake"""
    answer: Literal[1, 2, 3, 4]

Define response function for both OpenAI and Anthropic

In [8]:
def predict_headline(provider, model, prompt, date):

    print(f"Sending request for quiz: {date}, model: {model}")

    if provider == "openai":

        try:
            response = openai_client.responses.parse(
                model=model, 
                instructions=None,
                input=prompt,
                text_format=FakeHeadlineOutput
            )

            parsed = response.output_parsed

            print(f"Predicted answer: {parsed.answer if parsed is not None else None}")

            result = {
                "date": date,
                "prompt": prompt,
                "provider": provider,
                "model": response.model,
                "answer": parsed.answer if parsed is not None else None,
                "parse_success": parsed is not None,
                "response_id": response.id,
                "input_tokens": response.usage.input_tokens,
                "output_tokens": response.usage.output_tokens,
                "reasoning_tokens": (
                    response.usage.output_tokens_details.reasoning_tokens
                    if response.usage.output_tokens_details is not None
                    else None
                ),
                "total_tokens": response.usage.total_tokens,
                "raw_response": response,
                "error": None,
            }

        except Exception as e:
            result = {
                "date": date,
                "prompt": prompt,
                "provider": provider,
                "model": model,
                "answer": None,
                "parse_success": False,
                "response_id": None,
                "input_tokens": None,
                "output_tokens": None,
                "reasoning_tokens": None,
                "total_tokens": None,
                "raw_response": None,
                "error": repr(e),
            }

        return result

    elif provider == "anthropic":

        try:
            response = anthropic_client.messages.parse(
                model=model,
                max_tokens=1000,
                messages=[{"role": "user", "content": prompt}],
                output_format=FakeHeadlineOutput,
            )

            parsed = response.parsed_output

            print(f"Predicted answer: {parsed.answer if parsed is not None else None}")

            result = {
                "date": date,
                "prompt": prompt,
                "provider": provider,
                "model": response.model,
                "answer": parsed.answer if parsed is not None else None,
                "parse_success": parsed is not None,
                "response_id": response.id,
                "input_tokens": response.usage.input_tokens,
                "output_tokens": response.usage.output_tokens,
                "reasoning_tokens": (
                    response.usage.output_tokens_details.thinking_tokens
                    if response.usage.output_tokens_details is not None
                    else None
                ),
                "total_tokens": (
                    response.usage.input_tokens
                    + response.usage.output_tokens
                ),
                "raw_response": response,
                "error": None,
            }

        except Exception as e:
            result = {
                "date": date,
                "prompt": prompt,
                "provider": provider,
                "model": model,
                "answer": None,
                "parse_success": False,
                "response_id": None,
                "input_tokens": None,
                "output_tokens": None,
                "reasoning_tokens": None,
                "total_tokens": None,
                "raw_response": None,
                "error": repr(e),
            }

        return result

    else:
        raise ValueError("Valid provider not chosen")

### Testing

In [9]:
result = predict_headline("openai", "gpt-5-nano", test_set_frontier_LLM_ready[0]["prompt"], test_set_frontier_LLM_ready[0]["date"])

Sending request for quiz: 2019-11-23, model: gpt-5-nano
Predicted answer: 2


In [93]:
result

{'date': '2019-11-23',
 'prompt': 'Identify which of the four following headlines is fake:\n\n1: Taylor Swift fans boycott J.Crew over private equity ties\n\n2: Cambridge University students cry fowl over 17th century painting that upsets vegetarians\n\n3: Why the world is running out of sand\n\n4: Manager drives from New York to Pennsylvania to return keys\n\nAnswer: ',
 'provider': 'openai',
 'model': 'gpt-5-nano-2025-08-07',
 'answer': 2,
 'parse_success': True,
 'response_id': 'resp_0a63a6580178850b006ac65629047887d29b057144c1e29a9c',
 'input_tokens': 132,
 'output_tokens': 1197,
 'reasoning_tokens': 1152,
 'total_tokens': 1329,
 'raw_response': ParsedResponse[TypeVar](id='resp_0a63a6580178850b006ac65629047887d29b057144c1e29a9c', created_at=1791383081.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-5-nano-2025-08-07', object='response', output=[ResponseReasoningItem(id='rs_0a63a6580178850b006ac65640e61087d298cd77b4f80d180d', summary=[], type='reas

### Full run

In [10]:
results = []
for item in test_set_frontier_LLM_ready:
    for provider in models:
        for model in models[provider]:
            results.append(predict_headline(provider, model, item["prompt"], date=item["date"]))

    with open(f"{data_path}/frontier_llm_results.json", "w") as f:
        json.dump(
            [
                {k: v for k, v in result.items() if k != "raw_response"}
                for result in results
            ],
            f,
            indent=2
        )

Sending request for quiz: 2019-11-23, model: gpt-5-nano
Predicted answer: 2
Sending request for quiz: 2019-11-23, model: gpt-6-astra
Predicted answer: 1
Sending request for quiz: 2019-11-23, model: claude-haiku-4-5-20251001
Predicted answer: 2
Sending request for quiz: 2019-11-23, model: claude-fable-5-1
Predicted answer: 1
Sending request for quiz: 2020-01-18, model: gpt-5-nano
Predicted answer: 4
Sending request for quiz: 2020-01-18, model: gpt-6-astra
Predicted answer: 4
Sending request for quiz: 2020-01-18, model: claude-haiku-4-5-20251001
Predicted answer: 4
Sending request for quiz: 2020-01-18, model: claude-fable-5-1
Predicted answer: 4
Sending request for quiz: 2020-02-22, model: gpt-5-nano
Predicted answer: 3
Sending request for quiz: 2020-02-22, model: gpt-6-astra
Predicted answer: 1
Sending request for quiz: 2020-02-22, model: claude-haiku-4-5-20251001
Predicted answer: 1
Sending request for quiz: 2020-02-22, model: claude-fable-5-1
Predicted answer: 1
Sending request for qu

In [11]:
[x for x in results if x["error"] is not None]

[]

### Evaluate

In [12]:
true_answers = {
    item["date"]: item["answer"]
    for item in test_set
}

df = pd.DataFrame(results)

df["true_answer"] = df["date"].map(true_answers)
df["predicted_answer"] = df["answer"] - 1

df["correct"] = (
    df["predicted_answer"] == df["true_answer"]
)

summary = (
    df.groupby(["provider", "model"])
      .agg(
          correct=("correct", "sum"),
          total=("correct", "size"),
          parse_failures=("parse_success", lambda x: (~x).sum()),
      )
)

summary["accuracy"] = summary["correct"] / summary["total"]

summary

correct  total  parse_failures  accuracy
provider  model                                                              
anthropic claude-fable-5-1                59     67               0  0.880597
          claude-haiku-4-5-20251001       40     67               0  0.597015
openai    gpt-5-nano-2025-08-07           26     67               0  0.388060
          gpt-6-astra                     59     67               0  0.880597

In [13]:
frontier = df[
    df["model"].isin(["claude-fable-5-1", "gpt-6-astra"])
].pivot(index="date", columns="model", values="predicted_answer")

(frontier.iloc[:, 0] == frontier.iloc[:, 1]).mean()

np.float64(0.9104477611940298)

In [17]:
raw_results = []

for result in results:
    raw = result.copy()

    if raw["raw_response"] is not None:
        raw["raw_response"] = raw["raw_response"].to_dict(
            mode="json",
            warnings=False
        )

    raw_results.append(raw)

with open(f"{data_path}/frontier_llm_results_raw.json", "w") as f:
    json.dump(raw_results, f, indent=2)

In [2]:

with open("../data/frontier_llm_results.json", "r") as f:
    frontier_results = json.load(f)

frontier_df = pd.DataFrame(frontier_results)

# Check the clean rerun contains only successfully parsed answers
assert frontier_df["parse_success"].all()

# Convert human-readable answers 1–4 to canonical answers 0–3
frontier_df["prediction"] = frontier_df["answer"] - 1

# No directly comparable confidence measure is currently available
frontier_df["confidence"] = np.nan

# Save one standardised file per model
model_files = {
    "gpt-5-nano-2025-08-07": "predictions_gpt5_nano.csv",
    "gpt-6-astra": "predictions_gpt6_astra.csv",
    "claude-haiku-4-5-20251001": "predictions_claude_haiku.csv",
    "claude-fable-5-1": "predictions_claude_fable.csv",
}

for model, filename in model_files.items():
    model_results = frontier_df[frontier_df["model"] == model].copy()

    predictions = model_results[["date", "prediction", "confidence"]]

    predictions.to_csv(f"../data/{filename}", index=False)